# Lecture 20: Regression Trees

This notebook fits, tunes, and interprets a regression tree for housing prices.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeRegressor, plot_tree

from pathlib import Path


def find_repo_root(start=Path.cwd()):
    for path in [start, *start.parents]:
        if (path / "pyproject.toml").exists():
            return path
    raise RuntimeError("Could not find repository root")


ROOT = find_repo_root()
DATA = ROOT / "data" / "raw"


In [ ]:
housing = pd.read_csv(DATA / "housing_sales.csv")
features = ["size_sq_m", "rooms", "age_years", "renovation_score", "near_transit", "district"]
target = "price_k_eur"
X = housing[features]
y = housing[target]


In [ ]:
numeric = ["size_sq_m", "rooms", "age_years", "renovation_score", "near_transit"]
categorical = ["district"]
preprocess = ColumnTransformer(
    [
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical),
        ("num", "passthrough", numeric),
    ]
)
tree_pipe = Pipeline(
    [
        ("preprocess", preprocess),
        ("model", DecisionTreeRegressor(random_state=42)),
    ]
)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
param_grid = {
    "model__max_depth": [2, 3, 4, 5, 6, None],
    "model__min_samples_leaf": [5, 10, 20],
}
search = GridSearchCV(
    tree_pipe,
    param_grid=param_grid,
    cv=5,
    scoring="neg_root_mean_squared_error",
)
search.fit(X_train, y_train)
print(search.best_params_)
print(f"Best CV RMSE: {-search.best_score_:.2f}")


In [ ]:
best_tree = search.best_estimator_
test_pred = best_tree.predict(X_test)
print(f"Test RMSE: {mean_squared_error(y_test, test_pred) ** 0.5:.2f}")


In [ ]:
feature_names = best_tree.named_steps["preprocess"].get_feature_names_out()
tree_model = best_tree.named_steps["model"]
pd.Series(tree_model.feature_importances_, index=feature_names).sort_values(ascending=False)


In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))
plot_tree(
    tree_model,
    feature_names=feature_names,
    filled=True,
    max_depth=3,
    rounded=True,
    ax=ax,
)
ax.set_title("Top levels of the tuned regression tree")


## LLM Check

Ask an LLM to explain the first split. Verify that the explanation uses the actual threshold and does not overstate causality.
